# Defining Cofactors

This cookbook demonstrates how to define cofactors (any small molecules that are not the bound ligand) with 1) the ``openfe`` CLI during the planning phase and 2) directly using the Python API.

Similar to `ligand` definition, `cofactors` can be loaded from in SDF file format, and multiple molecules can be defined in a single file.

## 1) CLI: define cofactors during ``openfe plan-rbfe-network``

When using openfe's CLI, including cofactors in an RBFE calculation is as simple as passing in the ``-C`` argument with the path to the SDf file containing cofactors:

```bash
openfe plan-rbfe-network -M ligands.sdf -p protein.pdb -C cofactors.sdf
```

## 2) Python API: define cofactors in the ``ChemicalSystem``s 

If using the ``openfe`` Python API to set up an RBFE network, cofactors need to be included when constructing the ``ChemicalSystem``s.

In [1]:
import openfe

/Users/atravitz/micromamba/envs/openfe-notebooks/lib/python3.13/site-packages/mdtraj/core/selection.py:47: DeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat(cache_size_limit=304)


### Define the components (protein, ligands, cofactors)

Define the ligands, protein, and solvent in the same way as a typical RBFE calculation:

In [ ]:
from rdkit import Chem
from openff.toolkit import Molecule

# Create a list of OpenFF molecules
offmols = [
    Molecule.from_rdkit(m)
    for m in Chem.SDMolSupplier("assets/cofactors/eg5_ligands_positive.sdf", removeHs=False)
]

# Assign partial charges via AM1BCC
for mol in offmols:
    mol.assign_partial_charges('am1bcc')

# Now we convert them to SmallMoleculeComponent
ligands = [openfe.SmallMoleculeComponent.from_openff(mol) for mol in offmols]

# defaults are water with NaCl at 0.15 M
solvent = openfe.SolventComponent()

# load in the EG5 protein
protein = openfe.ProteinComponent.from_pdb_file("assets/cofactors/eg5_ligands_positive.sdfeg5_protein.pdb")

Load the cofactor(s) to be included in the simulations:

In [ ]:
# load in the cofactor from eg5_cofactors.sdf, here we know that there is only one cofactor in the file
# We also go through OpenFF to assign partial charges
offmol = Molecule.from_rdkit(Chem.MolFromMolFile("assets/cofactors/eg5_cofactor.sdf", removeHs=False))
offmol.assign_partial_charges('am1bcc', use_conformers=offmol.conformers)
cofactor = openfe.SmallMoleculeComponent.from_openff(offmol)
cofactor

### Create the Ligand Network

(This doesn't require any specific cofactor steps, but is included for completeness)

In [ ]:
mapper = openfe.LomapAtomMapper(max3d=1.0, element_change=False, threed=True)
scorer = openfe.lomap_scorers.default_lomap_score
network_planner = openfe.ligand_network_planning.generate_minimal_spanning_network

ligand_network = network_planner(
    ligands=ligands,
    mappers=mapper,
    scorer=scorer
)

### Define an RBFE Protocol

(This doesn't require any specific cofactor steps, but is included for completeness)

In [ ]:
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol

default_settings = RelativeHybridTopologyProtocol.default_settings()
protocol = RelativeHybridTopologyProtocol(default_settings)

### Create an ``AlchemicalNetwork`` that includes cofactors

When creating the legs of``AlchemicalNetwork``, you must explicitly define the cofactors as part of the complex legs' ``ChemicalSystem``s.

In [ ]:
transformations = []
for mapping in ligand_network.edges:
    for leg in ['solvent', 'complex']:
        # use the solvent and protein created above
        sysA_dict = {'ligand': mapping.componentA,
                     'solvent': solvent}
        sysB_dict = {'ligand': mapping.componentB,
                     'solvent': solvent}
        
        if leg == 'complex':
            sysA_dict['protein'] = protein
            sysB_dict['protein'] = protein
            # Add cofactors to both sysA and sysB 
            sysA_dict['cofactor'] = cofactor
            sysB_dict['cofactor'] = cofactor 
        
        # we don't have to name objects, but it can make things (like filenames) more convenient
        sysA = openfe.ChemicalSystem(sysA_dict, name=f"{mapping.componentA.name}_{leg}")
        sysB = openfe.ChemicalSystem(sysB_dict, name=f"{mapping.componentB.name}_{leg}")
        
        prefix = "rbfe_"  # prefix is only to exactly reproduce CLI
        
        transformation = openfe.Transformation(
            stateA=sysA,
            stateB=sysB,
            mapping={'ligand': mapping},
            protocol=protocol,  # use protocol created above
            name=f"{prefix}{sysA.name}_{sysB.name}"
        )
        transformations.append(transformation)

network = openfe.AlchemicalNetwork(transformations)

In [ ]:
network